# Vectorized Computation

## Definition
**Vectorization** means expressing operations on entire arrays (or batches) at once,
instead of iterating with Python `for` loops.

NumPy's vectorized operations:
- Execute in **compiled C/Fortran** code underneath
- Use **SIMD** (Single Instruction Multiple Data) CPU instructions for parallel math
- Avoid Python interpreter overhead per element
- Can be **10–1000x faster** than equivalent Python loops

### Core Principle
> Replace any Python loop that processes elements one-by-one with a NumPy array operation.

### AI/ML Relevance
Every deep learning forward pass, backprop, weight update, and loss computation
is a vectorized operation over batches.

In [ ]:
import numpy as np
import time

N = 1_000_000

# Approach 1: Pure Python loop
a = list(range(N))
b = list(range(N))

t0 = time.time()
c_loop = [a[i] + b[i] for i in range(N)]
t1 = time.time()
print(f'Python loop:   {t1-t0:.4f}s')

# Approach 2: NumPy vectorized
a_np = np.arange(N)
b_np = np.arange(N)

t0 = time.time()
c_np = a_np + b_np
t1 = time.time()
print(f'NumPy vector:  {t1-t0:.4f}s')

In [ ]:
# Vectorizing a custom function: sigmoid
def sigmoid_loop(x_list):
    return [1.0 / (1.0 + np.exp(-xi)) for xi in x_list]

def sigmoid_vec(x_arr):
    return 1.0 / (1.0 + np.exp(-x_arr))

N = 500_000
x_list = list(np.random.randn(N))
x_arr  = np.array(x_list)

t0 = time.time()
r1 = sigmoid_loop(x_list)
print(f'sigmoid loop:  {time.time()-t0:.4f}s')

t0 = time.time()
r2 = sigmoid_vec(x_arr)
print(f'sigmoid vec:   {time.time()-t0:.4f}s')

In [ ]:
# Vectorizing operations on rows/cols (avoid explicit loops)
mat = np.random.randn(1000, 100)

# Mean-center each row (Python loop approach — AVOID)
t0 = time.time()
result_loop = np.zeros_like(mat)
for i in range(mat.shape[0]):
    result_loop[i] = mat[i] - mat[i].mean()
print(f'Row centering loop: {time.time()-t0:.4f}s')

# Vectorized approach
t0 = time.time()
result_vec = mat - mat.mean(axis=1, keepdims=True)
print(f'Row centering vec:  {time.time()-t0:.4f}s')

print('Results match:', np.allclose(result_loop, result_vec))

In [ ]:
# np.vectorize — wraps a scalar function to work on arrays
# (NOTE: this is NOT a true vectorization — still Python speed!)
def clamp_scalar(x, lo, hi):
    return max(lo, min(hi, x))

clamp_vec = np.vectorize(clamp_scalar)
arr = np.array([-5.0, 0.5, 1.5, 2.0, 10.0])
print('Clamped [0,1]:', clamp_vec(arr, 0, 1))

# Better: use real vectorized ops
print('Clamped [0,1] numpy:', np.clip(arr, 0, 1))

In [ ]:
# np.clip — proper vectorized clamping
arr = np.array([-5, -1, 0, 2, 5, 10, 15])
clipped = np.clip(arr, 0, 10)
print('clip(0, 10):', clipped)

# ReLU
relu = np.maximum(arr, 0)
print('ReLU:', relu)

# Leaky ReLU
leaky = np.where(arr > 0, arr, 0.01 * arr)
print('LeakyReLU:', leaky)

In [ ]:
# Vectorized pairwise operations
# Cosine similarity between two matrices (batch)
def cosine_sim_matrix(A, B):
    # A: (n, d), B: (m, d)  -> result: (n, m)
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T

queries   = np.random.randn(10, 128)  # 10 query vectors
documents = np.random.randn(50, 128)  # 50 document vectors

sims = cosine_sim_matrix(queries, documents)
print('Cosine similarity matrix shape:', sims.shape)  # (10, 50)
print('Top doc for query 0:', np.argmax(sims[0]))